# Теория: Паттерн MVC

## 1. Что такое паттерн MVC?

`MVC` — это архитектурный паттерн, который делит приложение на три части:

- `Model` — данные и бизнес-логика;
- `View` — отображение данных;
- `Controller` — обработка действий пользователя и связь между `Model` и `View`.

Главная идея MVC: не смешивать данные, интерфейс и управление в одном месте.
Это делает код понятнее и удобнее в поддержке.


## 2. Цели и задачи паттерна Model-View-Controller

Цель MVC — разделить ответственность между частями приложения.

Что дает MVC:

- код легче читать и изменять;
- интерфейс можно менять отдельно от логики;
- бизнес-логику проще тестировать;
- проект проще расширять.

Без MVC код часто превращается в один большой блок, где ввод, вывод и логика перемешаны.


## 3. Model

### Что такое Model?

`Model` — это слой, который хранит данные и правила работы с ними.
Именно здесь находятся сущности приложения, проверки, расчеты и изменение состояния.

### Цели и задачи Model

- хранить данные приложения;
- реализовывать бизнес-логику;
- проверять корректность данных;
- изменять состояние через методы.

`Model` не должна знать, как данные показываются пользователю.


In [ ]:
from dataclasses import dataclass


# Product и CartModel относятся к Model,
# потому что они хранят данные и описывают правила работы с корзиной.
@dataclass
class Product:
    name: str
    price: int


class CartModel:
    def __init__(self):
        # Model хранит состояние приложения: каталог товаров и товары в корзине.
        self.products = {
            "apple": Product("Яблоко", 80),
            "banana": Product("Банан", 60),
            "coffee": Product("Кофе", 250),
        }
        self.items = []

    def add_item(self, product_code: str) -> None:
        # Здесь бизнес-логика: проверка существования товара перед добавлением.
        if product_code not in self.products:
            raise ValueError("Такого товара нет в каталоге")
        self.items.append(self.products[product_code])

    def total(self) -> int:
        # Model сама умеет считать итоговую сумму корзины.
        return sum(item.price for item in self.items)

    def item_names(self) -> list[str]:
        return [item.name for item in self.items]


model = CartModel()
model.add_item("apple")
model.add_item("coffee")
print(model.item_names())
print(model.total())


## 4. View

### Что такое View?

`View` — это слой, который показывает данные пользователю.
Он отвечает за внешний вид результата: текст, таблицу, HTML, JSON и другой формат вывода.

### Цели и задачи View

- отображать данные в понятном виде;
- форматировать вывод;
- показывать сообщения и ошибки.

`View` не должна содержать бизнес-логику и менять данные модели напрямую.


In [ ]:
# Это View, потому что класс только показывает данные пользователю.
# Он ничего не хранит и не изменяет состояние корзины.
class ConsoleCartView:
    @staticmethod
    def render_cart(items: list[str], total: int) -> None:
        print("Корзина:")
        if items:
            for item in items:
                print(f"- {item}")
        else:
            print("- пусто")
        print(f"Итого: {total} руб.")

    @staticmethod
    def render_error(message: str) -> None:
        print(f"Ошибка: {message}")


# View получает готовые данные и просто отображает их.
ConsoleCartView.render_cart(["Яблоко", "Кофе"], 330)


## 5. Controller

### Что такое Controller?

`Controller` — это слой, который принимает действие пользователя,
обращается к модели и передает результат в представление.

### Цели и задачи Controller

- принимать команды, запросы и события;
- вызывать нужные методы `Model`;
- передавать результат в `View`;
- связывать все части приложения в единый сценарий.

`Controller` не должен хранить основную бизнес-логику вместо модели.


In [ ]:
# Это Controller, потому что он принимает действие,
# обращается к Model и передает результат во View.
class CartController:
    def __init__(self, model: CartModel, view: ConsoleCartView):
        self.model = model
        self.view = view

    def add_product(self, product_code: str) -> None:
        # Controller не хранит данные сам, а координирует работу слоев.
        try:
            self.model.add_item(product_code)
            self.view.render_cart(self.model.item_names(), self.model.total())
        except ValueError as error:
            self.view.render_error(str(error))


# Здесь controller связывает model и view в единый сценарий работы.
controller = CartController(CartModel(), ConsoleCartView())
controller.add_product("banana")
controller.add_product("coffee")
controller.add_product("water")


## 6. Примеры использования паттерна MVC

### Веб-приложения

- `Model` — данные и работа с базой;
- `Controller` — обработка HTTP-запроса;
- `View` — HTML-страница или JSON-ответ.

### Консольные программы

- `Model` хранит данные;
- `Controller` обрабатывает команды;
- `View` выводит результат в консоль.

### GUI и desktop-приложения

- `View` — окна, кнопки, формы;
- `Controller` — реакция на действия пользователя;
- `Model` — данные и правила работы приложения.

### Коротко

MVC подходит там, где нужно отдельно управлять данными, интерфейсом и логикой взаимодействия.


In [ ]:
# Эта схема показывает поток данных в MVC: действие идет в Controller,
# затем он вызывает Model и передает результат во View.
events = [
    "Пользователь -> Controller: add_product('apple')",
    "Controller -> Model: add_item('apple')",
    "Model -> Controller: корзина обновлена",
    "Controller -> View: render_cart(items, total)",
    "View -> Пользователь: показать обновленную корзину",
]

for step in events:
    print(step)